# Masters London 2026 — Team EDA

Focuses on **recent form only** — change `MIN_DATE` in the first code cell.

Sections:
1. Setup & sample sizes
2. Map win rate heatmap
3. Permaban inference
4. Attack vs Defence per map
5. Overall stats table
6. Head-to-head matrix
7. Recent form (rolling win rate)
8. Single-team deep dive

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
from collections import defaultdict
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False

# Change this to adjust the lookback window
MIN_DATE = '2025-09-01'   # current season  <-- good default
# MIN_DATE = '2026-01-01' # last ~5 months (small samples)
# MIN_DATE = '2025-06-01' # last ~12 months

with open('../data/processed/raw_matches.json') as f:
    ALL_MATCHES = json.load(f)

print(f'Total matches: {len(ALL_MATCHES)}')
print(f'Lookback: {MIN_DATE} onwards')

# LEVIATÁN is stored with U+00C1 (capital A with acute) in the scraped JSON
LEVIATAN = 'LEVIAT' + chr(0x00C1) + 'N'

ML_TEAMS = [
    'G2 Esports',  LEVIATAN,        'NRG',
    'Team Heretics','Team Vitality', 'FUT Esports',
    'Paper Rex',   'FULL SENSE',    'Global Esports',
    'EDward Gaming','Xi Lai Gaming', 'Dragon Ranger Gaming',
]

DISPLAY = {t: t for t in ML_TEAMS}
DISPLAY[LEVIATAN] = 'LEVIATÁN'

REGION = {
    'G2 Esports': 'Americas', LEVIATAN: 'Americas',        'NRG': 'Americas',
    'Team Heretics': 'EMEA',  'Team Vitality': 'EMEA',     'FUT Esports': 'EMEA',
    'Paper Rex': 'Pacific',   'FULL SENSE': 'Pacific',     'Global Esports': 'Pacific',
    'EDward Gaming': 'China', 'Xi Lai Gaming': 'China',    'Dragon Ranger Gaming': 'China',
}

REGION_COLORS = {
    'Americas': '#e74c3c', 'EMEA': '#3498db',
    'Pacific':  '#2ecc71', 'China': '#f39c12',
}

CURRENT_POOL = ['Haven', 'Lotus', 'Split', 'Pearl', 'Ascent', 'Breeze', 'Fracture']

def get_matches(team, min_date=MIN_DATE):
    out = []
    for m in ALL_MATCHES:
        if not m.get('date') or m['date'] < min_date: continue
        if m['team_a'] == team:   out.append((m, 'a'))
        elif m['team_b'] == team: out.append((m, 'b'))
    return sorted(out, key=lambda x: x[0]['date'])

print()
print(f"{'Team':<28} {'Region':<10} {'Series':>7}")
print('-' * 48)
for t in ML_TEAMS:
    n = len(get_matches(t))
    flag = '  low' if n < 10 else ''
    print(f"{DISPLAY[t]:<28} {REGION[t]:<10} {n:>7}{flag}")


## 2. Map Win Rate Heatmap

In [ ]:
def build_map_stats(team):
    raw = defaultdict(lambda: dict(played=0, wins=0, rwr=[], atk=[], dfn=[]))
    for match, side in get_matches(team):
        for mp in match.get('maps', []):
            name = mp.get('map', '')
            if not name or name == 'unknown': continue
            s = raw[name]
            s['played'] += 1
            if mp.get('winner_side') == side: s['wins'] += 1
            for key, lst in [('round_win_rate_' + side, s['rwr']),
                              ('atk_win_rate_' + side,   s['atk']),
                              ('def_win_rate_' + side,   s['dfn'])]:
                if key in mp: lst.append(mp[key])
    result = {}
    for name, s in raw.items():
        p = s['played']
        result[name] = dict(
            played=p, wins=s['wins'], losses=p - s['wins'],
            win_rate=s['wins'] / p if p else 0.5,
            avg_rwr=np.mean(s['rwr']) if s['rwr'] else 0.5,
            avg_atk=np.mean(s['atk']) if s['atk'] else 0.25,
            avg_def=np.mean(s['dfn']) if s['dfn'] else 0.25,
        )
    return result

ALL_STATS = {t: build_map_stats(t) for t in ML_TEAMS}

def ban_scores(team):
    total = len(get_matches(team))
    if total == 0: return {m: 0 for m in CURRENT_POOL}
    s = ALL_STATS[team]
    return {m: s.get(m, {}).get('played', 0) / total for m in CURRENT_POOL}

print('Map stats built.')


In [ ]:
wr_data, pl_data = [], []
for t in ML_TEAMS:
    row_wr, row_pl = [], []
    for m in CURRENT_POOL:
        s = ALL_STATS[t].get(m)
        row_wr.append(s['win_rate'] if s and s['played'] >= 3 else float('nan'))
        row_pl.append(s['played']   if s else 0)
    wr_data.append(row_wr); pl_data.append(row_pl)

wr_df = pd.DataFrame(wr_data, index=[DISPLAY[t] for t in ML_TEAMS], columns=CURRENT_POOL)
pl_df = pd.DataFrame(pl_data, index=[DISPLAY[t] for t in ML_TEAMS], columns=CURRENT_POOL)

fig, ax = plt.subplots(figsize=(13, 7))
im = ax.imshow(wr_df.values, cmap='RdYlGn', vmin=0.25, vmax=0.75, aspect='auto')
ax.set_xticks(range(len(CURRENT_POOL)))
ax.set_xticklabels(CURRENT_POOL, fontsize=12, fontweight='bold')
ax.set_yticks(range(len(ML_TEAMS)))
ax.set_yticklabels([f"{DISPLAY[t]}  [{REGION[t]}]" for t in ML_TEAMS], fontsize=10)
for i, t in enumerate(ML_TEAMS):
    for j, mp in enumerate(CURRENT_POOL):
        wr = wr_df.iloc[i, j]; pl = pl_df.iloc[i, j]
        if wr != wr:
            ax.text(j, i, f'n={pl}', ha='center', va='center', fontsize=8, color='#888')
        else:
            ax.text(j, i, f'{wr:.0%}\n({pl}g)', ha='center', va='center',
                    fontsize=8.5, fontweight='bold',
                    color='white' if wr < 0.35 or wr > 0.68 else 'black')
plt.colorbar(im, ax=ax, label='Map Win Rate', shrink=0.7)
ax.set_title(f'Map Win Rates (since {MIN_DATE})', fontsize=13, pad=14)
plt.tight_layout(); plt.show()
print('Green=dominant  Red=weak  n=X means fewer than 3 maps played')


## 3. Permaban Inference

Shorter bar = team almost never plays this map = likely permaban.

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(18, 13))
axes = axes.flatten()
for idx, team in enumerate(ML_TEAMS):
    ax = axes[idx]
    scores = ban_scores(team)
    maps_s = sorted(CURRENT_POOL, key=lambda m: scores[m])
    vals   = [scores[m] for m in maps_s]
    colors = plt.cm.RdYlGn(np.linspace(0.15, 0.85, len(maps_s)))
    bars = ax.barh(maps_s, vals, color=colors, edgecolor='#ccc', linewidth=0.4)
    if vals and vals[-1] > 0 and vals[0] < vals[-1] * 0.4:
        ax.text(vals[0] + 0.005, 0, 'permaban?', va='center',
                fontsize=7.5, color='firebrick', fontweight='bold')
    for bar, m in zip(bars, maps_s):
        ms = ALL_STATS[team].get(m, {})
        pl = ms.get('played', 0)
        wr = ms.get('win_rate')
        label = str(pl) if not wr or pl == 0 else f'{pl} ({wr:.0%})'
        ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height() / 2,
                label, va='center', fontsize=7)
    n_series = len(get_matches(team))
    ax.set_title(f"{DISPLAY[team]}  [{REGION[team]}]\n{n_series} series",
                 fontsize=9, fontweight='bold')
    ax.set_xlabel('Maps played per series', fontsize=7)
    ax.set_xlim(0, max(vals) * 1.5 if any(v > 0 for v in vals) else 1)
    ax.tick_params(axis='y', labelsize=9); ax.tick_params(axis='x', labelsize=7)
plt.suptitle(
    f'Permaban Inference  (since {MIN_DATE})\n'
    'Shorter bar = rarely played = likely permabanned  |  Label: maps (win rate)',
    fontsize=12, y=1.01)
plt.tight_layout(); plt.show()


## 4. Attack vs Defence Strength per Map

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(18, 14))
axes = axes.flatten()
for idx, team in enumerate(ML_TEAMS):
    ax = axes[idx]
    s = ALL_STATS[team]
    maps_w = [m for m in CURRENT_POOL if m in s and s[m]['played'] >= 3]
    if not maps_w:
        ax.text(0.5, 0.5, 'insufficient data', ha='center', va='center',
                transform=ax.transAxes, color='gray')
        ax.set_title(DISPLAY[team], fontsize=9); continue
    x = np.arange(len(maps_w)); w = 0.35
    ax.bar(x - w/2, [s[m]['avg_atk'] for m in maps_w], w, label='Attack (T)',   color='tomato',    alpha=0.85)
    ax.bar(x + w/2, [s[m]['avg_def'] for m in maps_w], w, label='Defence (CT)', color='steelblue', alpha=0.85)
    ax.axhline(0.25, color='black', linestyle='--', linewidth=0.8)
    ax.set_xticks(x); ax.set_xticklabels(maps_w, rotation=35, ha='right', fontsize=8)
    ax.set_ylim(0, 0.55); ax.set_ylabel('Round win rate', fontsize=7)
    ax.set_title(f"{DISPLAY[team]}  [{REGION[team]}]", fontsize=9, fontweight='bold')
    if idx == 0: ax.legend(fontsize=7.5, loc='upper right')
plt.suptitle(
    f'Attack vs Defence per Map  (since {MIN_DATE})\n'
    'Red=T-side  |  Blue=CT-side  |  dashed=average (0.25)',
    fontsize=12, y=1.01)
plt.tight_layout(); plt.show()


## 5. Overall Stats Table

In [ ]:
rows = []
for team in ML_TEAMS:
    matches = get_matches(team)
    n = len(matches)
    results = [int(m['winner'] == (0 if s == 'a' else 1)) for m, s in matches]
    w = sum(results)
    streak, streak_str = 0, '-'
    if results:
        last = results[-1]
        for r in reversed(results):
            if r == last: streak += 1
            else: break
        streak_str = f"W{streak}" if last == 1 else f"L{streak}"
    last5 = ''.join('W' if r == 1 else 'L' for r in results[-5:])
    ps = {m: ALL_STATS[team][m] for m in CURRENT_POOL
          if m in ALL_STATS[team] and ALL_STATS[team][m]['played'] >= 3}
    best  = max(ps, key=lambda m: ps[m]['win_rate']) if ps else '-'
    worst = min(ps, key=lambda m: ps[m]['win_rate']) if ps else '-'
    bs = ban_scores(team)
    pban = min(bs, key=bs.get) if bs else '-'
    rows.append({
        'Team':    DISPLAY[team],
        'Region':  REGION[team],
        'Series':  n,
        'W%':      f'{w/n:.0%}' if n else '-',
        'Last 5':  last5,
        'Streak':  streak_str,
        'Best map':  f"{best} ({ps[best]['win_rate']:.0%})" if best != '-' else '-',
        'Worst map': f"{worst} ({ps[worst]['win_rate']:.0%})" if worst != '-' else '-',
        'Likely ban': f"{pban} ({ALL_STATS[team].get(pban, {}).get('played', 0)}g)",
    })
df = pd.DataFrame(rows).set_index('Team')
print(df.to_string())


## 6. Head-to-Head Between ML Teams

In [ ]:
n = len(ML_TEAMS); ti = {t: i for i, t in enumerate(ML_TEAMS)}
h2h_w = np.zeros((n, n)); h2h_cnt = np.zeros((n, n), dtype=int)
for m in ALL_MATCHES:
    if m.get('date', '') < MIN_DATE: continue
    ta, tb = m['team_a'], m['team_b']
    if ta not in ti or tb not in ti: continue
    i, j = ti[ta], ti[tb]
    h2h_cnt[i, j] += 1; h2h_cnt[j, i] += 1
    aw = (m['winner'] == 0)
    h2h_w[i, j] += int(aw); h2h_w[j, i] += int(not aw)
with np.errstate(invalid='ignore'):
    h2h_rate = np.where(h2h_cnt > 0, h2h_w / h2h_cnt, float('nan'))
short = ['G2','LEVI','NRG','HER','VIT','FUT','PRX','FSN','GE','EDG','XLG','DRG']
import numpy.ma as ma
fig, ax = plt.subplots(figsize=(11, 9))
masked = ma.masked_invalid(h2h_rate)
im = ax.imshow(masked, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
ax.set_xticks(range(n)); ax.set_xticklabels(short, fontsize=9)
ax.set_yticks(range(n)); ax.set_yticklabels([DISPLAY[t] for t in ML_TEAMS], fontsize=9)
ax.set_title(f'H2H Win Rate (since {MIN_DATE})\nRow vs column  |  dash = never played', fontsize=12)
for i in range(n):
    for j in range(n):
        if i == j: continue
        cnt = h2h_cnt[i, j]
        if cnt == 0:
            ax.text(j, i, '-', ha='center', va='center', fontsize=12, color='#aaa')
        else:
            wr = h2h_rate[i, j]
            ax.text(j, i, f'{wr:.0%}\n({cnt})', ha='center', va='center', fontsize=7.5,
                    color='white' if (wr < 0.25 or wr > 0.75) else 'black')
plt.colorbar(im, ax=ax, label='Win rate (row team)', shrink=0.7)
plt.tight_layout(); plt.show()
print('Most cross-region matchups are dash (never played in this window) -- normal.')


## 7. Recent Form

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.flatten()
for idx, team in enumerate(ML_TEAMS):
    ax = axes[idx]
    matches = get_matches(team)
    if not matches:
        ax.text(0.5, 0.5, 'no data', ha='center', va='center',
                transform=ax.transAxes, color='gray')
        ax.set_title(DISPLAY[team], fontsize=9); continue
    results = np.array([int(m['winner'] == (0 if s == 'a' else 1))
                        for m, s in matches], dtype=float)
    window  = min(8, len(results))
    rolling = pd.Series(results).rolling(window, min_periods=3).mean().values
    color = REGION_COLORS[REGION[team]]
    x = np.arange(len(rolling))
    ax.plot(x, rolling, color=color, linewidth=2)
    ax.fill_between(x, rolling, 0.5, where=(rolling >= 0.5), alpha=0.18, color='green')
    ax.fill_between(x, rolling, 0.5, where=(rolling <  0.5), alpha=0.18, color='red')
    ax.axhline(0.5, color='gray', linewidth=0.8, linestyle='--')
    for k, (res, rv) in enumerate(zip(results, rolling)):
        if rv == rv:
            ax.scatter(k, rv, color='green' if res == 1 else 'red', s=18, zorder=5, alpha=0.8)
    w = int(results.sum())
    ax.set_title(f"{DISPLAY[team]}  [{REGION[team]}]\n"
                 f"{w}W-{len(results)-w}L  ({results.mean():.0%})",
                 fontsize=8.5, fontweight='bold')
    ax.set_ylim(-0.05, 1.05)
    ax.set_ylabel(f'{window}-series rolling W%', fontsize=7)
    ax.set_xlabel('oldest to newest', fontsize=7)
    ax.tick_params(labelsize=7)
plt.suptitle(f'Recent Form  (since {MIN_DATE})\n'
             'Green dot=Win  Red dot=Loss  Shading=above/below .500',
             fontsize=12, y=1.01)
plt.tight_layout(); plt.show()


## 8. Single-team Deep Dive

Change `TEAM` to any of the 12 teams.

In [ ]:
TEAM = 'Paper Rex'  # change this to any team name
                    # For LEVIATÁN, type 'LEVIATÁN' or 'LEVIATAN' -- handled automatically

if 'LEVIAT' in TEAM.upper():
    TEAM = LEVIATAN

stats   = ALL_STATS[TEAM]
matches = get_matches(TEAM)
results = [int(m['winner'] == (0 if s == 'a' else 1)) for m, s in matches]
n = len(results); w = sum(results)

print(f"\n{DISPLAY[TEAM]}  |  {REGION[TEAM]}  |  since {MIN_DATE}")
print(f"{w}W - {n-w}L  ({w/n:.1%})" if n else 'No data')
print('Last 5: ' + ' '.join('W' if r else 'L' for r in results[-5:]))
print()
print(f"  {'Map':<12} {'Played':>8} {'W-L':>6} {'Win%':>6} {'Rnd%':>7} {'Atk%':>7} {'Def%':>7}")
print('  ' + '-' * 58)
for m in sorted(CURRENT_POOL,
                key=lambda x: stats.get(x, {'win_rate': -1})['win_rate'],
                reverse=True):
    s = stats.get(m)
    if not s:
        print(f"  {m:<12} {'0':>8}  -- no data")
    else:
        wl = f"{s['wins']}-{s['losses']}"
        print(f"  {m:<12} {s['played']:>8} {wl:>6} {s['win_rate']:>6.0%}"
              f" {s['avg_rwr']:>7.0%} {s['avg_atk']:>7.0%} {s['avg_def']:>7.0%}")

bs = ban_scores(TEAM)
pban = min(bs, key=bs.get)
ppick = max(bs, key=bs.get)
print(f"\n  Likely permaban : {pban}  "
      f"({ALL_STATS[TEAM].get(pban, {}).get('played', 0)} maps played)")
print(f"  Likely pick     : {ppick}  "
      f"({ALL_STATS[TEAM].get(ppick, {}).get('played', 0)} maps  |  "
      f"{ALL_STATS[TEAM].get(ppick, {}).get('win_rate', 0):.0%} WR)")
